In [ ]:
import json
import matplotlib.pyplot as plt

In [ ]:
test_file = "../results/2026-03-04/rq-dgemm69a8659c0b313e.json"
f = open(test_file,'r')
data = json.load(f)


In [ ]:
data.keys()

In [ ]:
data["runs"][0].keys()

In [ ]:
def simple_plot(stats):
  vec_stats = 0
  for stat in stats:
    if isinstance(stat.get("value"), list):
      vec_stats+=1

  fig, axs = plt.subplots(1, vec_stats, figsize=(5*vec_stats, 5))
  count = 0
  for stat in stats:
     if isinstance(stat.get("value"), list):
      plt.sca(axs[count])
      count+=1
      plt.plot(stat["value"])
      plt.title("Evolution of "+stat["name"]+" across repetitions")
      plt.ylabel(stat["name"]+" "+stat["unit"])
      plt.xlabel("Repetitions")
      plt.tight_layout()

      
  plt.show()

In [ ]:
#for run in data["runs"]:
  #for run_result in run["results"]:
    
   #simple_plot(run_result["stats"])


In [ ]:
def plot_suite_result(suite_preset,results,metric_name="median"):
  x = []
  y = []
  metric_unit = ""
  print(suite_preset)
  axe = suite_preset['options']['SingleAxe']
  for result in results:
    print(result["options"][axe["interface_name"]])
    x.append(result["options"][axe["interface_name"]][axe["option_name"]])
    for metric in result["stats"]:
      if metric["name"] == metric_name:
        y.append(metric["value"])
        metric_unit = metric["unit"]
  plt.plot(x,y)
  plt.grid()  
  plt.title(suite_preset["preset_name"] + " : "+suite_preset["description"])
  plt.xlabel(axe["option_name"])
  plt.ylabel(metric_name + f" ( {metric_unit} )")
  plt.show()
  print(x,y)


In [ ]:
def find_preset(Result,preset_name):
  for preset in Result["presets"]:
    if preset["preset_name"] == preset_name:
      return preset

In [ ]:
for run in data["runs"]:
  plot_suite_result(find_preset(data,run["recipe"]["suite"]["preset"]),run["results"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots()

# Plot some data (colors might vary, but legend will use custom ones)
ax.plot(np.arange(5), np.arange(5), color='gray', label='Gray line')
ax.plot(np.arange(5), np.arange(5)[::-1], color='orange', label='Orange line')
ax.legend()


In [ ]:
from dataLoader import *

In [ ]:
@dataclass(frozen=True)
class InnerPreset:
  desc:str
  options:OptionContainer
  
@dataclass(frozen=True)
class OuterPreset:
  impl:str
  preset_name:str

In [ ]:
def build_preset_dict(presets)->Dict[OuterPreset,InnerPreset]:
  preset_dict = {}
  for preset in presets:
    preset_dict[OuterPreset(preset["implementation"],preset["preset_name"])] = InnerPreset(preset["description"],preset["options"])
  return preset_dict

In [ ]:
def populate_impl(preset_dict:Dict[OuterPreset,InnerPreset],impl_json)->Impl:
  inner = preset_dict[OuterPreset(impl_json["name"],impl_json["preset"])]
  return Impl(impl_json["name"],impl_json["preset"],inner.desc,inner.options)

In [ ]:
def build_recipe(recipe,preset_dict:Dict[OuterPreset,InnerPreset]):
  backend = populate_impl(preset_dict,recipe["backend"])
  benchmark = populate_impl(preset_dict,recipe["benchmark"])
  case_ = populate_impl(preset_dict,recipe["case"])
  stats = populate_impl(preset_dict,recipe["stats"])
  stopping = populate_impl(preset_dict,recipe["stopping_criterion"])
  suite = None
  if "suite" in recipe:
    suite = populate_impl(preset_dict,recipe["suite"])
  return Recipe(backend,benchmark,case_,stats,stopping,suite)

In [ ]:
def build_device(device)->GpuDevice:
  return GpuDevice(device["name"])

In [ ]:
def build_benchmark_run_setup(run,global_metadata:ResultFileMetadata,preset_dict:Dict[OuterPreset,InnerPreset])->BenchmarkRunSetup:
  recipe = build_recipe(run["recipe"],preset_dict)
  device = build_device(run["device"])
  return BenchmarkRunSetup(global_metadata.datetime,global_metadata.git_version,global_metadata.baseliner_version,run["id"],device,recipe)

In [ ]:
test_file = "../results/2026-03-04/rq-dgemm69a8659c0b313e.json"
f = open(test_file,'r')
data = json.load(f)


In [ ]:
metadata = ResultFileMetadata(data["datetime"],data["git_version"],data["baseliner_version"])
dict_preset = build_preset_dict(data["presets"])

for run in data["runs"]:
  print(build_benchmark_run_setup(run,metadata,dict_preset))

In [ ]:
for run in data["runs"]:
  benchmark_impl = populate_impl
